## Testdaten
Für das Projekt müssen zumindest einmalig Testdaten generiert werden, die zum RAG-Datensatz passen. Das werde ich wieder mit einem LLM erstellen lassen, ich denke, die Aufgabe ist nicht schwer wenn es nur dedizierte Daten erhält. Die Testmenge wird überschaubar bleiben und kann in handarbeit evaluiert werden.

Es soll zwei Datensätze an Testdaten geben. Zum eine Fragen, für deren Beantwortung ein bestimmter Chunk gefunden werden muss, zum anderen solche, deren Antworten über mehrere Chunks verteilt ist.

Für die Singe-Chunk-Fragen werden nur technische Daten verwendet. Um diese filtern zu können muss auf die Metadaten zugegriffen werden, die ein JSON-Objekt mit den Daten enthalten. Eine Möglichkeit ist, den Typen in eine eigene Spalte zu schreiben, ist letztlich ja pro Chunk typisch.

Aus den Datensätzen sollen zufällig Datensätze gezogen und an das LLM zur Generierung der Testfragen gesendet werden.

In [8]:
import os
import re
import json
import random
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv
from collections import defaultdict


load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=120000)

In [9]:
# Requestfunktion
def agent_request(system_promt, schema, content):

    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            'type': 'json_object',
            'json_schema': schema,
            'strict': True
        }
    )

    return response

## Dataprep

In [10]:
# Daten laden
with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    products_chunked = [json.loads(line) for line in f]

descs_chunks = [item for item in products_chunked if item.get('metadata', {}).get('chunk_type') == 'desc']
specs_chunks = [item for item in products_chunked if item.get('metadata', {}).get('chunk_type') == 'spec']

print(len(descs_chunks))
print(len(specs_chunks))
    

152
1277


### Single Chunk Questions

In [11]:
with open('../data/prompts/test_single_chunk_agent.md', 'r') as f:
    specs_prompt = f.read()

with open('../data/prompts/test_single_chunk_schema.json', 'r') as f:
    specs_schema = json.load(f)

In [12]:
specs_quests = []
specs_samples = random.sample(specs_chunks, k=25)

# print(specs_samples)
for spec in tqdm(specs_samples, total=len(specs_samples)):

    print("=" * 50)
    print("LLM Response:")

    response = agent_request(specs_prompt, specs_schema, spec['document'])
    # Model sendet gelegentlich ein ``` was raus muss  
    content = response.choices[0].message.content.strip()
    content = re.sub(r'\s*```$', '', content)

    print(response)
    print("=" * 50)
    
    for question in json.loads(content):
        specs_quests.append({
            'product_id': spec['metadata']['product_id'],
            'question': question,
            'answer': spec['document'],
            'chunk_id': spec['id']
        })
    
with open('../data/tests/specs_question.json', 'w', encoding='utf-8') as f:
    json.dump(specs_quests, f, ensure_ascii=False, indent=2)

# print(specs_quests)

  0%|          | 0/25 [00:00<?, ?it/s]

LLM Response:


  4%|▍         | 1/25 [00:00<00:22,  1.05it/s]

id='7a92f2b5f6214c85b76595a49ab629b4' object='chat.completion' model='mistral-medium-2508' usage=UsageInfo(prompt_tokens=679, completion_tokens=59, total_tokens=738, prompt_audio_seconds=Unset()) created=1762978123 choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='["Welche Art von Schloss hat der HMTvh 1511?", "Kann der HMTvh 1511 per Fernbedienung geöffnet werden?", "Verfügt der HMTvh 1511 über ein elektronisches Schloss?"]', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
LLM Response:


  8%|▊         | 2/25 [00:02<00:27,  1.20s/it]

...

id='5299829b82ad4cffbc25ed132132a7fd' object='chat.completion' model='mistral-medium-2508' usage=UsageInfo(prompt_tokens=688, completion_tokens=55, total_tokens=743, prompt_audio_seconds=Unset()) created=1762978156 choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='["Welche Art von Display hat der HMTvh 1511?", "Verfügt der HMTvh 1511 über ein Touch-Display?", "Wie groß ist das Display des HMTvh 1511?"]', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
LLM Response:


 96%|█████████▌| 24/25 [00:35<00:01,  1.26s/it]

id='86ef4edb625c4726b30776cf23d1e13a' object='chat.completion' model='mistral-medium-2508' usage=UsageInfo(prompt_tokens=688, completion_tokens=54, total_tokens=742, prompt_audio_seconds=Unset()) created=1762978157 choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='["Welche Wärmeabgabe hat der Kirsch FROSTER LABEX-530 ULTIMATE?", "Wie viel Watt Wärmeabgabe hat der Kirsch FROSTER LABEX-530 ULTIMATE?"]', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]
LLM Response:


100%|██████████| 25/25 [00:36<00:00,  1.48s/it]

id='fd357febd4394a4ea3f85d0cf21a3aff' object='chat.completion' model='mistral-medium-2508' usage=UsageInfo(prompt_tokens=686, completion_tokens=71, total_tokens=757, prompt_audio_seconds=Unset()) created=1762978158 choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='[\n    "Welchen Temperaturbereich hat der Kirsch LABO-125?",\n    "Kann der Kirsch LABO-125 auf eine Temperatur von +10 °C eingestellt werden?",\n    "Was ist die niedrigste einstellbare Temperatur des Kirsch LABO-125?"\n]', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')]


### Multi Chunk Questions

In [13]:
with open('../data/prompts/test_multi_chunk_agent.md', 'r') as f:
    descs_prompt = f.read()

with open('../data/prompts/test_multi_chunk_schema.json', 'r') as f:
    descs_schema = json.load(f)

In [14]:
# Gruppieren
all_products = defaultdict(list)
for chunk in descs_chunks:
    product_id = chunk.get('metadata', {}).get('product_id')
    all_products[product_id].append(chunk)


# Auswahl an Produkten
selected_products = random.sample(list(all_products.keys()), k=3)

# Quests
descs_quests = []
for product_id in selected_products:
    product_chunks = all_products[product_id]
    descs_samples = []
    
    if len(product_chunks) >= 2:
        num_chunks = min(len(product_chunks), 3)
        sampled_chunks = random.sample(product_chunks, k=num_chunks)
        
        # Nur die document Strings extrahieren
        doc_strings = [c['document'] for c in sampled_chunks]
        descs_samples.append(doc_strings)

    print(descs_samples)
    response = agent_request(descs_prompt, descs_schema, json.dumps(descs_samples))
    
    for question in json.loads(response.choices[0].message.content):
        descs_quests.append({
            'product_id': spec['metadata']['product_id'],
            'question': question,
            'chunk_id': spec['id']
        })
        

with open('../data/tests/multi_question.json', 'w', encoding='utf-8') as f:
    json.dump(descs_quests, f, ensure_ascii=False, indent=2)

[['Der Liebherr LGUex1500 ist ein explosionsgeschützter Gefrierschrank (Kühlgerät, Kühlaggregat), der die EU-Richtlinie 2014/34/EU (ATEX) erfüllt. Mit der Klassifizierung II 3G Ex nA II T6 ist das Gerät ideal zur Lagerung von explosiven und leicht entzündlichen Stoffen in geschlossenen Behältnissen geeignet. Dies macht den Gefrierschrank besonders interessant für Sonderlaboratorien und die chemische Industrie. Das Gerät ist unterbaufähig und verfügt über drei Schubfächer und einen Korb.', 'Der Liebherr LGUex1500 ist ein hochwertiges Kühlgerät (Kühlschrank, Kühlaggregat), das speziell für Laboratorien und die chemische Industrie konzipiert wurde. Das Gerät ist mit einer Komfortelektronik ausgestattet, die eine präzise Temperaturregelung ermöglicht. Die Temperatur im Inneren wird exakt auf dem eingestellten Wert gehalten, was für die Lagerung sensibler Substanzen entscheidend ist. Die folierte Anzeige und Tastatur gewährleisten hohe Hygienestandards, da sie leicht abwaschbar sind.', 'Der